In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

# =============================================================================
# INPUT / OUTPUT
# =============================================================================

HAZARD_CSV = Path("data/hazard.csv")
EXPOSURE_CSV = Path("data/exposure.csv")
VULNERABILITY_CSV = Path("data/vulnerability.csv")

OUTPUT_CSV = Path("data/final_risk_score.csv")

# =============================================================================
# LOAD DATA
# =============================================================================

hazard = pd.read_csv(HAZARD_CSV)[["district", "timeperiod", "heat_hazard"]]
exposure = pd.read_csv(EXPOSURE_CSV)[["district", "timeperiod", "exposure"]]
vulnerability = pd.read_csv(VULNERABILITY_CSV)[["district", "timeperiod", "vulnerability_class"]]

df = (
    hazard
    .merge(exposure, on=["district", "timeperiod"], how="inner")
    .merge(vulnerability, on=["district", "timeperiod"], how="inner")
)

print("Merged shape:", df.shape)

# =============================================================================
# WEIGHTS
# =============================================================================

weights = np.array([4, 2, 1], dtype=float)
weights = weights / weights.sum()

criteria = np.array([1, 1, 1])  # all are benefit criteria

# =============================================================================
# TOPSIS CLASS (INLINE)
# =============================================================================

class Topsis:
    def __init__(self, evaluation_matrix, weight_matrix, criteria):
        self.evaluation_matrix = np.array(evaluation_matrix, dtype="float")
        self.weight_matrix = np.array(weight_matrix, dtype="float")
        self.weight_matrix = self.weight_matrix / self.weight_matrix.sum()
        self.criteria = np.array(criteria, dtype="float")

        self.row_size = len(self.evaluation_matrix)
        self.col_size = len(self.evaluation_matrix[0])

    def step_2(self):
        self.normalized = np.copy(self.evaluation_matrix)
        sq = np.zeros(self.col_size)

        for i in range(self.row_size):
            for j in range(self.col_size):
                sq[j] += self.evaluation_matrix[i, j] ** 2

        for i in range(self.row_size):
            for j in range(self.col_size):
                self.normalized[i, j] = self.evaluation_matrix[i, j] / np.sqrt(sq[j])

    def step_3(self):
        self.weighted = self.normalized * self.weight_matrix

    def step_4(self):
        self.best = np.zeros(self.col_size)
        self.worst = np.zeros(self.col_size)

        for i in range(self.col_size):
            if self.criteria[i]:
                self.best[i] = np.max(self.weighted[:, i])
                self.worst[i] = np.min(self.weighted[:, i])
            else:
                self.best[i] = np.min(self.weighted[:, i])
                self.worst[i] = np.max(self.weighted[:, i])

    def step_5(self):
        self.d_best = np.sqrt(((self.weighted - self.best) ** 2).sum(axis=1))
        self.d_worst = np.sqrt(((self.weighted - self.worst) ** 2).sum(axis=1))

    def step_6(self):
        self.score = self.d_worst / (self.d_best + self.d_worst)

    def calc(self):
        self.step_2()
        self.step_3()
        self.step_4()
        self.step_5()
        self.step_6()
        return self.score

# =============================================================================
# MONTH-WISE TOPSIS
# =============================================================================

results = []

for month, group in df.groupby("timeperiod"):

    temp = group.copy()

    matrix = temp[[
        "heat_hazard",
        "vulnerability_class",
        "exposure"
    ]].values

    model = Topsis(matrix, weights, criteria)
    temp["topsis_score"] = model.calc()

    results.append(temp)

result = pd.concat(results, ignore_index=True)

# =============================================================================
# NORMALIZATION + BINNING
# =============================================================================

def zscore(x):
    std = x.std(ddof=0)
    if std == 0:
        return pd.Series(0, index=x.index)
    return (x - x.mean()) / std


result["risk_z"] = result.groupby("timeperiod")["topsis_score"].transform(zscore)

def classify(z):
    if z <= -1.5:
        return 1
    elif z <= -0.5:
        return 2
    elif z <= 0.5:
        return 3
    elif z <= 1.5:
        return 4
    else:
        return 5

result["risk_class"] = result["risk_z"].apply(classify)

# =============================================================================
# SAVE OUTPUT
# =============================================================================

output_cols = [
    "district",
    "timeperiod",
    "heat_hazard",
    "vulnerability_class",
    "exposure",
    "topsis_score",
    "risk_z",
    "risk_class"
]

result[output_cols].to_csv(OUTPUT_CSV, index=False)

print("Saved:", OUTPUT_CSV)

print("\nRisk distribution:")
print(result["risk_class"].value_counts().sort_index())

Merged shape: (690, 5)
Saved: data/final_risk_score.csv

Risk distribution:
risk_class
1     34
2    183
3    275
4    164
5     34
Name: count, dtype: int64


In [6]:
from pathlib import Path

print("CWD:", Path.cwd())
print("Hazard exists:", Path("data/hazard.csv").exists())
print("Exposure exists:", Path("data/exposure.csv").exists())
print("Vulnerability exists:", Path("data/vulnerability.csv").exists())

CWD: /home/root_1/Documents/CDL/repos/IDS-DRR_Heat/Odisha
Hazard exists: False
Exposure exists: False
Vulnerability exists: False
